#RAG_BASED_CHATBOT_USING_LANGCHAIN_+_HUGGING_FACE_+_FAISS

In [ ]:
##Rag -> document training -> answer

In [ ]:
## 1. Install required libraries
!pip install -q -U langchain langchain-community langchain-huggingface \
    langchain-text-splitters sentence-transformers faiss-cpu \
    pypdf transformers accelerate

In [ ]:
# 2. Import libraries
from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

In [ ]:
# 3. Upload PDF
print("Upload your PDF file:")

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("\nUploaded PDF:", pdf_file)

#here, I have uploaded by old heart disease dataset file

Upload your PDF file:


Saving Dataset Heart Disease.pdf to Dataset Heart Disease.pdf

Uploaded PDF: Dataset Heart Disease.pdf


In [ ]:
# 4. Load PDF
from langchain_community.document_loaders import PyPDFLoader

pdf_file = "/content/Dataset Heart Disease.pdf"

loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("Number of pages:", len(documents))

print(documents[0].page_content)

Number of pages: 29
age sex chest pain tyresting bps cholesterol fasting bloodresting ecg max heart raexercise angoldpeak ST slope target
0 40 1 2 140 289.0 0 0 172 0 0.0 1 0
1 49 0 3 160 180.0 0 0 156 0 1.0 2 1
2 37 1 2 130 283.0 0 1 98 0 0.0 1 0
3 48 0 4 138 214.0 0 0 108 1 1.5 2 1
4 54 1 3 150 195.0 0 0 122 0 0.0 1 0
5 39 1 3 120 339.0 0 0 170 0 0.0 1 0
6 45 0 2 130 237.0 0 0 170 0 0.0 1 0
7 54 1 2 110 208.0 0 0 142 0 0.0 1 0
8 37 1 4 140 207.0 0 0 130 1 1.5 2 1
9 48 0 2 120 284.0 0 0 120 0 0.0 1 0
10 37 0 3 130 211.0 0 0 142 0 0.0 1 0
11 58 1 2 136 164.0 0 1 99 1 2.0 2 1
12 39 1 2 120 204.0 0 0 145 0 0.0 1 0
13 49 1 4 140 234.0 0 0 140 1 1.0 2 1
14 42 0 3 115 211.0 0 1 137 0 0.0 1 0
15 54 0 2 120 273.0 0 0 150 0 1.5 2 0
16 38 1 4 110 196.0 0 0 166 0 0.0 2 1
17 43 0 2 120 201.0 0 0 165 0 0.0 1 0
18 60 1 4 100 248.0 0 0 125 0 1.0 2 1
19 36 1 2 120 267.0 0 0 160 0 3.0 2 1
20 43 0 1 100 223.0 0 0 142 0 0.0 1 0
21 44 1 2 120 184.0 0 0 142 0 1.0 2 0
22 49 0 2 124 201.0 0 0 164 0 0.0 1 0


In [ ]:
# 5. Split PDF into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of text chunks:", len(chunks))

Number of text chunks: 110


In [ ]:
# 6. Create Embeddings
print("\nCreating embeddings...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings created!")



Creating embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created!


In [ ]:
# 7. Create FAISS Vector Database
print("\nCreating vector database...")

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created!")


Creating vector database...
FAISS vector database created!


In [ ]:
# 8. Create Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever ready!")

Retriever ready!


In [ ]:
# 9. Load Hugging Face LLM
print("\nLoading LLM...")

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.3
)

llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline
)

print("LLM loaded successfully!")


Loading LLM...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

... (4 outputs left out)

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'Apertus1p5TextForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BailingHybridForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BeamForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dor

LLM loaded successfully!


In [ ]:
# 10. RAG Function
def rag_chat(question):

    # Retrieve relevant documents
    docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # Create prompt
    prompt = f"""
Answer the question using only the information
provided in the context.

If the answer is not present in the context,
say "I could not find the answer in the document."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
    answer = llm.invoke(prompt)

    return answer


In [ ]:
# 11. Start Chatbot
print("\n========================================")
print("       RAG CHATBOT IS READY!")
print("========================================")
print("Ask questions about your PDF.")
print("Type 'exit' to stop.\n")

while True:

    question = input("You: ")

    if question.lower() == "exit":
        print("Bot: Goodbye! 👋")
        break

    answer = rag_chat(question)

    print("\nBot:", answer)
    print("\n" + "-" * 60 + "\n")


       RAG CHATBOT IS READY!
Ask questions about your PDF.
Type 'exit' to stop.

You: exit
Bot: Goodbye! 👋
